# Replicating CEBRA Paper Style — Decoding Static Image Identity from VISp

This notebook adapts the structure and low-level training style of the CEBRA visual-cortex replication workflow to the Allen Visual Behavior Neuropixels dataset.

Instead of decoding natural-movie frames from pseudomice, we use repeated static image presentations from a single VISp-rich session and ask whether CEBRA can learn a latent space that preserves **image identity** from VISp population activity.


## Setup

In [1]:
import sys
import torch

print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: /opt/conda/bin/python
Torch version: 2.10.0+cu128
CUDA available: True
Torch CUDA version: 12.8
CUDA device count: 1
GPU: NVIDIA A30


In [2]:
import os
os.environ["CEBRA_DATADIR"] = os.getcwd()

print("cwd:", os.getcwd())
print("allen_vbn_cache exists:", os.path.exists("allen_vbn_cache"))

cwd: /home/adl007/BIPN-162-Final-Project/BIPN-162-Final-Project
allen_vbn_cache exists: True


In [3]:
import sys
!{sys.executable} -m pip install -q 'cebra[datasets,demos]' matplotlib seaborn scikit-learn numpy torch allensdk
print('Done.')

# %pip install -U "numpy>=1.25"
# %pip install --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==2.2.2"

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.1.0 requires fsspec[http]<=2024.9.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
dask-expr 1.0.11 requires pandas>=2, but you have pandas 1.5.3 which is incompatible.
Done.


---

In [4]:
# PART 1 IMPORTS

# Importing standard python libraries for data manipulation, plotting, machine learning tasks, 
# AllenSDK access, and CEBRA's low-level API.

import os, sys, itertools, random
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression

from allensdk.brain_observatory.behavior.behavior_project_cache.behavior_neuropixels_project_cache import (
    VisualBehaviorNeuropixelsProjectCache,
)

import cebra
import cebra.data
import cebra.datasets

# Using GPU if it is available because CEBRA training is much faster on GPU.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'CEBRA version: {cebra.__version__}')

seed = 333
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

Using device: cuda
CEBRA version: 0.6.0


### Part 1b — Load and Visualize the Data

In [ ]:
# Single-session VISp static-image dataset
#
# The original paper uses 10 movie repeats. To stay close to that split logic, we:
#   1. choose one VISp-rich Allen VBN session,
#   2. keep the 8 most common static image identities,
#   3. downsample to a count divisible by 10,
#   4. split each image into 10 equal pseudo-repeats,
#   5. use pseudo-repeats 1-9 for training/validation and pseudo-repeat 10 for test.
#
# Continuous labels (DINO analogue):
#   Since static images do not come with frame-wise DINO features, we compute
#   template-level image feature vectors by flattening each warped template image
#   and projecting the pixel vectors with PCA. These template features then play
#   the role of continuous visual-content labels for CEBRA-Behavior.

cache_dir = "./allen_vbn_cache"
os.makedirs(cache_dir, exist_ok=True)

cache = VisualBehaviorNeuropixelsProjectCache.from_s3_cache(cache_dir=cache_dir)

sessions = cache.get_ecephys_session_table()
units = cache.get_unit_table()

candidate_sessions = [1108334384, 1087723305, 1043752325]
visp_counts = (
    units.loc[units["structure_acronym"] == "VISp"]
    .groupby("ecephys_session_id")
    .size()
    .rename("n_visp_units")
)

candidate_summary = pd.DataFrame(index=candidate_sessions)
candidate_summary["n_visp_units"] = visp_counts.reindex(candidate_sessions).fillna(0).astype(int)
display(candidate_summary)

session_id = candidate_summary["n_visp_units"].idxmax()
print("Selected session:", session_id)

session = cache.get_ecephys_session(ecephys_session_id=session_id)
templates = session.stimulus_templates.copy()

# Build balanced repeated-image dataset
stim = session.stimulus_presentations.copy()
stim = stim.dropna(subset=["image_name", "start_time"]).copy()

if "omitted" in stim.columns:
    stim = stim[~stim["omitted"]].copy()

if "is_change" in stim.columns:
    stim = stim[~stim["is_change"]].copy()

stim = stim[stim["image_name"].astype(str).str.startswith("im")].copy()

top_images = stim["image_name"].value_counts().index[:8]
stim = stim[stim["image_name"].isin(top_images)].copy()

# To mimic the paper's 10-repeat logic, keep a count divisible by 10.
min_count = stim["image_name"].value_counts().min()
usable_count = (min_count // 10) * 10
if usable_count < 10:
    raise ValueError("Not enough repeated presentations per image to build 10 pseudo-repeats.")

stim_balanced = (
    stim.groupby("image_name", group_keys=False)
    .sample(n=usable_count, random_state=seed)
    .sort_values(["image_name", "start_time"])
    .copy()
)

presentations_per_block = usable_count // 10
stim_balanced["repeat_block"] = -1

for img in sorted(stim_balanced["image_name"].unique()):
    img_idx = stim_balanced.index[stim_balanced["image_name"] == img].to_numpy()
    for block_id in range(10):
        block_slice = img_idx[block_id * presentations_per_block:(block_id + 1) * presentations_per_block]
        stim_balanced.loc[block_slice, "repeat_block"] = block_id

stim_balanced = stim_balanced.sort_values(["repeat_block", "image_name", "start_time"]).copy()

# Select good VISp units from the global unit table, which includes structure_acronym.
session_units = units[units["ecephys_session_id"] == session_id].copy()
if "quality" in session_units.columns:
    visp_units = session_units[
        (session_units["structure_acronym"] == "VISp") &
        (session_units["quality"] == "good")
    ].copy()
else:
    visp_units = session_units[
        session_units["structure_acronym"] == "VISp"
    ].copy()

# Build presentation-aligned firing-rate tensor
bin_size = 0.025
window_start = 0.0
window_end = 0.25

bins = np.arange(window_start, window_end + bin_size, bin_size)
bin_centers = bins[:-1] + bin_size / 2
bins_per_presentation = len(bin_centers)

spike_times = session.spike_times
unit_ids = visp_units.index.to_numpy()

presentation_times = stim_balanced["start_time"].to_numpy()
presentation_labels = stim_balanced["image_name"].to_numpy()

n_presentations = len(presentation_times)
n_units = len(unit_ids)
n_bins = len(bin_centers)

firing_rates = np.zeros((n_presentations, n_units, n_bins), dtype=np.float32)

for i, t0 in enumerate(presentation_times):
    for j, uid in enumerate(unit_ids):
        spikes = spike_times[uid]
        rel_spikes = spikes - t0
        counts, _ = np.histogram(rel_spikes, bins=bins)
        firing_rates[i, j, :] = counts.astype(np.float32) / bin_size

# Build continuous template-feature labels (static-image analogue of DINO features)
image_names = sorted(stim_balanced["image_name"].unique())
template_pixels = np.stack([
    templates.loc[img, "warped"].astype(np.float32).ravel()
    for img in image_names
])

template_pixels_z = StandardScaler().fit_transform(template_pixels)
template_feature_dim = min(8, template_pixels_z.shape[0], template_pixels_z.shape[1])
template_features = PCA(n_components=template_feature_dim, random_state=seed).fit_transform(template_pixels_z)

feature_by_image = {img: template_features[i] for i, img in enumerate(image_names)}

label_encoder = LabelEncoder()
label_encoder.fit(image_names)
stim_balanced["image_id"] = label_encoder.transform(stim_balanced["image_name"])

def build_split_namespace(split_df):
    idx = split_df.index.to_numpy()
    x_3d = firing_rates[idx]
    n_present = x_3d.shape[0]

    neural = np.transpose(x_3d, (0, 2, 1)).reshape(n_present * bins_per_presentation, n_units)
    continuous = np.repeat(
        np.stack([feature_by_image[img] for img in split_df["image_name"]]),
        bins_per_presentation,
        axis=0
    )
    discrete = np.repeat(split_df["image_id"].to_numpy(), bins_per_presentation)
    block = np.repeat(split_df["repeat_block"].to_numpy(), bins_per_presentation)

    return SimpleNamespace(
        neural=torch.tensor(neural, dtype=torch.float32),
        index=torch.tensor(continuous, dtype=torch.float32),
        discrete=torch.tensor(discrete, dtype=torch.long),
        block=torch.tensor(block, dtype=torch.long),
        presentation_ids=split_df["image_id"].to_numpy(),
        presentation_labels=split_df["image_name"].to_numpy(),
        presentation_blocks=split_df["repeat_block"].to_numpy(),
    )

# Train = pseudo-repeats 1-9 (block 0-8), test = pseudo-repeat 10 (block 9)
train_df = stim_balanced[stim_balanced["repeat_block"] < 9].sort_values(
    ["repeat_block", "image_name", "start_time"]
).copy()

test_df = stim_balanced[stim_balanced["repeat_block"] == 9].sort_values(
    ["image_name", "start_time"]
).copy()

visp_train = build_split_namespace(train_df)
visp_test = build_split_namespace(test_df)

cortex = "VISp"
num_neurons = visp_train.neural.shape[1]

print(f'\nDataset shapes:')
print(f'  VISp train neural: {tuple(visp_train.neural.shape)} = (9 pseudo-repeats × presentations × {bins_per_presentation} bins, {num_neurons} neurons)')
print(f'  VISp train labels: {tuple(visp_train.index.shape)} = (T, {template_feature_dim}) template feature vectors')
print(f'  VISp test neural: {tuple(visp_test.neural.shape)}')
print(f'  Number of image identities: {len(image_names)}')
print(f'  Presentations per image used: {usable_count} = 10 pseudo-repeats × {presentations_per_block} presentations each')

,n_visp_units
1108334384,220
1087723305,189
1043752325,231


Selected session: 1043752325


/home/adl007/.local/lib/python3.11/site-packages/hdmf/spec/namespace.py:620: UserWarning: Ignoring the following cached namespace(s) because another version is already loaded:
core - cached version: 2.6.0-alpha, loaded version: 2.7.0
The loaded extension(s) may not be compatible with the cached extension(s) in the file. Please check the extension documentation and ignore this warning if these versions are compatible.
  self.warn_for_ignored_namespaces(ignored_namespaces)


In [ ]:
# Visualizing Raw Neural Activity
# Before training, we visualize raw presentation-aligned VISp activity to confirm
# the data structure and show how neural responses vary across aligned image windows.

fig = plt.figure(figsize=(10, 5))

ax1 = plt.subplot(1, 2, 1)
# Show the first 80 presentations (80 × 10 bins) from the train split.
raw_matrix = visp_train.neural.cpu().numpy()[:80 * bins_per_presentation].T
ax1.imshow(raw_matrix, aspect='auto', cmap='gray_r')
ax1.set_ylabel('# Neurons')
ax1.set_xlabel('Aligned time bins')
ax1.set_title('VISp firing rates (presentation aligned)')

ax2 = plt.subplot(1, 2, 2)
# Average firing rate for each image identity across all aligned presentations.
mean_by_image = []
for img in image_names:
    img_idx = train_df.index[train_df["image_name"] == img].to_numpy()
    mean_by_image.append(firing_rates[img_idx].mean(axis=(0, 2)))
mean_by_image = np.stack(mean_by_image)

ax2.imshow(mean_by_image, aspect='auto', cmap='gray_r')
ax2.set_yticks(np.arange(len(image_names)))
ax2.set_yticklabels(image_names)
ax2.set_xlabel('# Neurons')
ax2.set_title('Mean VISp response by image identity')

plt.tight_layout()
plt.savefig('p1_01_raw_activity_static_visp.png', dpi=120, bbox_inches='tight')
plt.show()

**Figure 1: Raw Data Visualization of VISp Static-Image Responses**  
The two panels above show the raw neural input used for the adapted VISp analysis. The left panel visualizes presentation-aligned VISp firing rates across neurons and short post-stimulus time windows, confirming that the notebook is working with time-resolved Neuropixels responses rather than averaged summaries. The right panel averages these aligned responses within each image identity, showing that different static images evoke different coarse response patterns in VISp. This replaces the Ca-versus-Neuropixels comparison in the original replication notebook, because the present notebook focuses on a single recording modality and a different stimulus regime.


### Part 1b Ctd -  Visualize Static Image Template Features

The original replication notebook visualizes DINO frame features because those continuous visual-content vectors are the labels guiding CEBRA-Behavior. Static image presentations do not come with frame-wise DINO features, so here we use PCA features of the warped stimulus templates as the closest visual-content analogue. Each unique image identity therefore gets one low-dimensional template feature vector that is repeated across all presentations of that image.


In [ ]:
# Static image template features as behavioral labels (DINO analogue)

# Create a t-SNE model that reduces template features to 2 dimensions. With only
# 8 image identities, the visualization shows one point per image rather than a
# dense movie trajectory.
template_tsne = TSNE(
    n_components=2,
    init='pca',
    learning_rate='auto',
    perplexity=min(5, len(image_names) - 1),
    random_state=seed
)

template_tsne_viz = template_tsne.fit_transform(template_features)

fig = plt.figure(figsize=(6, 5))
sc = plt.scatter(
    template_tsne_viz[:, 0],
    template_tsne_viz[:, 1],
    c=np.arange(len(image_names)),
    cmap='magma',
    s=80
)

for i, img in enumerate(image_names):
    plt.text(template_tsne_viz[i, 0], template_tsne_viz[i, 1], f' {img}', fontsize=9)

plt.colorbar(sc, label='Image index')
plt.axis('off')
plt.title('Static image template features (t-SNE 2D)\nColor = image identity')
plt.tight_layout()
plt.savefig('p1_02_template_feature_tsne.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Template feature dimension: {template_features.shape[1]} (one vector per static image)')
print('These vectors are the static-image analogue of the DINO labels fed to CEBRA alongside neural activity.')

**Figure 2 - Static Image Template Features**  
The plot above is the static-image analogue of the DINO figure from the original replication notebook. Each point corresponds to one unique flashed image template, positioned according to a low-dimensional feature vector computed from the image itself. Because this dataset contains only repeated static images rather than a continuously evolving movie, the plot shows a small set of discrete points instead of a smooth movie trajectory. Even so, these template feature vectors still play the same conceptual role as DINO in the original notebook: they provide continuous visual-content labels that CEBRA can use to define positive pairs beyond raw time adjacency.


### Part 1c — Define Helper Functions (Official Paper API)

In [ ]:
# CEBRA Model Architecture and Helper Functions
'''
As in the replication notebook, we use the low-level solver API rather than the
compact sklearn wrapper. This keeps the training format, comments, and execution
flow as close as possible to the official paper-style notebook while still
working on a different dataset.

Because we only have one recording modality (VISp Neuropixels-like aligned spike
matrices), we only need a single_session_solver here.
'''

def single_session_solver(data_loader, **kwargs):
    '''
    Build a SingleSessionSolver for one recording modality.
    '''
    norm = kwargs['distance'] != 'euclidean'
    data_loader.to(kwargs['device'])

    model = cebra.models.init(
        kwargs['model_architecture'],
        data_loader.dataset.input_dimension,
        kwargs['num_hidden_units'],
        kwargs['output_dimension'],
        norm
    ).to(kwargs['device'])

    data_loader.dataset.configure_for(model)

    criterion = (
        cebra.models.InfoMSE(temperature=kwargs['temperature'])
        if kwargs['distance'] == 'euclidean'
        else cebra.models.InfoNCE(temperature=kwargs['temperature'])
    )

    optimizer = torch.optim.Adam(
        itertools.chain(model.parameters(), criterion.parameters()),
        lr=kwargs['learning_rate']
    )

    return cebra.solver.SingleSessionSolver(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        tqdm_on=kwargs['verbose']
    )


@torch.no_grad()
def get_emissions(model, dataset):
    '''
    Run the trained encoder forward on a dataset to get the embedding (no gradients).
    "Emissions" = encoder output = latent embedding vectors.
    Returns a NumPy array of shape (T, output_dimension).
    '''
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    dataset.configure_for(model)
    return model(dataset[torch.arange(len(dataset))].to(device)).cpu().numpy()


def compute_emissions_single(solver, dataset):
    return get_emissions(solver.model, dataset)


def average_bins_per_presentation(feature, bins_per_presentation):
    '''
    Average consecutive bins back to one vector per presentation.
    This is the static-image analogue of averaging 4 NP bins into one movie frame.
    '''
    if isinstance(feature, torch.Tensor):
        feature = feature.cpu().numpy()
    return feature.reshape(-1, bins_per_presentation, feature.shape[-1]).mean(axis=1)


def labels_per_presentation(labels, bins_per_presentation):
    '''
    Convert repeated bin-level labels back to one label per presentation.
    '''
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    return labels.reshape(-1, bins_per_presentation)[:, 0]


def static_image_id_decode(train_fs, train_labels, test_fs, test_labels,
                           bins_per_presentation=10, decoder='knn'):
    '''
    Decode static image IDs from raw activity or CEBRA embeddings.

    Mirrors allen_frame_id_decode() from the replication notebook:
      - train on pseudo-repeats 1-8,
      - validate hyperparameters on pseudo-repeat 9,
      - retrain on pseudo-repeats 1-9,
      - test on pseudo-repeat 10.

    Returns exact image classification accuracy (% correct class).
    '''
    train_fs = average_bins_per_presentation(train_fs, bins_per_presentation)
    test_fs = average_bins_per_presentation(test_fs, bins_per_presentation)

    train_labels = labels_per_presentation(train_labels, bins_per_presentation)
    test_labels = labels_per_presentation(test_labels, bins_per_presentation)

    params = (np.power(np.linspace(1, 10, 5, dtype=int), 2)
              if decoder == 'knn' else np.logspace(-9, 3, 5))
    errs = []

    # Pseudo-repeats 1-8 train, pseudo-repeat 9 validate
    valid_idx = int(len(train_fs) / 9 * 8)

    for n in params:
        clf = (KNeighborsClassifier(n_neighbors=n, metric='cosine')
               if decoder == 'knn' else GaussianNB(var_smoothing=n))

        clf.fit(train_fs[:valid_idx], train_labels[:valid_idx])
        pred = clf.predict(train_fs[valid_idx:])
        errs.append((pred != train_labels[valid_idx:]).sum())

    best_n = params[np.argmin(errs)]

    test_clf = (KNeighborsClassifier(n_neighbors=best_n, metric='cosine')
                if decoder == 'knn' else GaussianNB(var_smoothing=best_n))
    test_clf.fit(train_fs, train_labels)
    pred = test_clf.predict(test_fs)

    acc = (pred == test_labels).mean() * 100
    return pred, (pred != test_labels), acc


print('Success! All helper functions defined.')

### Part 1d — Train CEBRA-Behavior Models  

In [ ]:
# Training CEBRA-Behavior Models [Adapted Fig. 4]

# ARCHITECTURE:
# VISp static-image data uses 'offset1-model' because we have a single
# time-aligned neural stream and one feature vector per aligned bin.
# Unlike the original movie notebook, there is no 120 Hz vs 30 Hz mismatch
# that requires resample1-model.

# HYPERPARAMETERS:
# We keep the paper-style settings and comments as closely as possible.
# num_steps is reduced relative to the original movie notebook because the
# static-image dataset is smaller and single-modality only.

train_steps = 5000
seed = 333
cortex = 'VISp'
num_neurons = visp_train.neural.shape[1]
num_images = len(image_names)

# Build TensorDataset objects using the official low-level CEBRA data API.
# continuous = template feature vectors (DINO analogue)
# discrete   = image identity labels
visp_train_ds = cebra.data.datasets.TensorDataset(
    visp_train.neural,
    continuous=visp_train.index,
    discrete=visp_train.discrete
)

visp_test_ds = cebra.data.datasets.TensorDataset(
    visp_test.neural,
    continuous=visp_test.index,
    discrete=visp_test.discrete
)

def make_loader(ds):
    '''Data loaders must be re-created after each .fit() call because they are consumed after one training run.'''
    return cebra.data.ContinuousDataLoader(
        ds,
        num_steps=train_steps,
        batch_size=512,
        conditional='time_delta',
        time_offset=1
    )

# MODEL A: 8D scatter plot visualization
# We train an 8D model purely for visualization purposes.
print('\nTraining 8D VISp model (for scatter plots, adapted Fig. 4)...')

visp_l = make_loader(visp_train_ds)

m_visp_8d = single_session_solver(
    visp_l,
    model_architecture='offset1-model',
    distance='cosine',
    num_hidden_units=128,
    output_dimension=8,
    verbose=False,
    device=DEVICE,
    temperature=1,
    learning_rate=3e-4
)
m_visp_8d.fit(visp_l)

emb_visp_8d = compute_emissions_single(m_visp_8d, visp_train_ds)
emb_visp_test_8d = compute_emissions_single(m_visp_8d, visp_test_ds)

print(f'  VISp 8D train: {emb_visp_8d.shape}  VISp 8D test: {emb_visp_test_8d.shape}')

# MODEL B: 128D — decoding accuracy
print('\nTraining 128D VISp model (for decoding bar chart, adapted Fig. 4)...')

visp_l = make_loader(visp_train_ds)

m_visp_128d = single_session_solver(
    visp_l,
    model_architecture='offset1-model',
    distance='cosine',
    num_hidden_units=128,
    output_dimension=128,
    verbose=True,
    device=DEVICE,
    temperature=1,
    learning_rate=3e-4
)
m_visp_128d.fit(visp_l)

emb_visp_128d = compute_emissions_single(m_visp_128d, visp_train_ds)
emb_visp_test_128d = compute_emissions_single(m_visp_128d, visp_test_ds)

print(f'  VISp 128D train: {emb_visp_128d.shape}  VISp 128D test: {emb_visp_test_128d.shape}')

In [ ]:
# VISUALIZE 8D EMBEDDINGS  [Adapted Fig. 4c,d,f,g]

# Instead of four modality/joint panels, we visualize 2D slices of the single
# VISp 8D embedding for train and held-out test presentations.
# Color = image identity.

emb_visp_8d_present = average_bins_per_presentation(emb_visp_8d, bins_per_presentation)
emb_visp_test_8d_present = average_bins_per_presentation(emb_visp_test_8d, bins_per_presentation)

train_colors = train_df["image_id"].to_numpy()
test_colors = test_df["image_id"].to_numpy()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

panels = [
    (axes[0, 0], emb_visp_8d_present[:, [0, 1]], train_colors, f'VISp — train (dims 1,2) — adapted Fig. 4'),
    (axes[0, 1], emb_visp_8d_present[:, [2, 3]], train_colors, f'VISp — train (dims 3,4) — adapted Fig. 4'),
    (axes[1, 0], emb_visp_test_8d_present[:, [0, 1]], test_colors, f'VISp — held-out test (dims 1,2)'),
    (axes[1, 1], emb_visp_test_8d_present[:, [2, 3]], test_colors, f'VISp — held-out test (dims 3,4)'),
]

for ax, emb, colors, title in panels:
    ax.scatter(emb[:, 0], emb[:, 1], c=colors, cmap='magma', s=10)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle(
    f'CEBRA-Behavior 8D Embeddings — {cortex}, {num_neurons} neurons, seed={seed}\n'
    'Color = static image identity. Structured clusters = image information encoded.',
    fontsize=12,
    y=1.01
)
plt.tight_layout()
plt.savefig('fig4_embeddings_8D_static_visp.png', dpi=150, bbox_inches='tight')
plt.show()

**Figure 3 - 8D Visualizations in 2D Slices**  
The four scatter plots above show 2D slices of the 8D CEBRA-Behavior embedding learned from VISp static-image data. Each point represents one image presentation after averaging the short aligned time bins back into a single presentation-level vector. The color indicates which static image was shown. Instead of the movie-loop geometry seen in the original replication notebook, the adapted VISp analysis should produce clusters corresponding to repeated image identities. The held-out test panels are especially important because they show whether this structure generalizes beyond the pseudo-repeats used during training.


### Part 1e — Decode Static Image IDs  [Adapted Fig. 4 bar chart]

Metric: **% of held-out test presentations classified as the correct image identity**.  
Decoder hyperparameter (k or var\_smoothing) is selected on pseudo-repeat 9, after training on pseudo-repeats 1–8, and the final model is then evaluated on pseudo-repeat 10.

| Condition | Meaning |
|-----------|---------|
| kNN on raw VISp | baseline on raw presentation-aligned activity |
| Naive Bayes on raw VISp | probabilistic baseline on raw activity |
| PCA + kNN | linear low-dimensional baseline |
| CEBRA-VISp (128D) → kNN | adapted paper-style embedding decoder |

This step evaluates how well the trained CEBRA embedding preserves image identity compared with simpler baselines on the completely held-out pseudo-repeat.


In [ ]:
# Static Image Decoding (Adapted Fig. 4 bar chart)

# The static_image_id_decode() function handles:
#   presentation averaging (10 aligned bins -> 1 presentation vector)
#   hyperparameter selection using pseudo-repeat 9
#   test on pseudo-repeat 10 only

print('Decoding held-out static image presentations...')

# BASELINE 1: kNN on raw VISp activity
pred_knn, _, acc_knn = static_image_id_decode(
    visp_train.neural, visp_train.discrete,
    visp_test.neural, visp_test.discrete,
    bins_per_presentation=bins_per_presentation,
    decoder='knn'
)

# BASELINE 2: Naive Bayes on raw VISp activity
pred_bayes, _, acc_bayes = static_image_id_decode(
    visp_train.neural, visp_train.discrete,
    visp_test.neural, visp_test.discrete,
    bins_per_presentation=bins_per_presentation,
    decoder='bayes'
)

# BASELINE 3: PCA + kNN on raw VISp activity
train_raw_present = average_bins_per_presentation(visp_train.neural, bins_per_presentation)
test_raw_present = average_bins_per_presentation(visp_test.neural, bins_per_presentation)
train_present_labels = labels_per_presentation(visp_train.discrete, bins_per_presentation)
test_present_labels = labels_per_presentation(visp_test.discrete, bins_per_presentation)

pca = PCA(n_components=min(8, train_raw_present.shape[1]), random_state=seed)
train_pca = pca.fit_transform(train_raw_present)
test_pca = pca.transform(test_raw_present)

valid_idx = int(len(train_pca) / 9 * 8)
params = np.power(np.linspace(1, 10, 5, dtype=int), 2)
errs = []

for n in params:
    clf = KNeighborsClassifier(n_neighbors=n, metric='cosine')
    clf.fit(train_pca[:valid_idx], train_present_labels[:valid_idx])
    pred = clf.predict(train_pca[valid_idx:])
    errs.append((pred != train_present_labels[valid_idx:]).sum())

best_n_pca = params[np.argmin(errs)]
pca_clf = KNeighborsClassifier(n_neighbors=best_n_pca, metric='cosine')
pca_clf.fit(train_pca, train_present_labels)
pred_pca = pca_clf.predict(test_pca)
acc_pca = (pred_pca == test_present_labels).mean() * 100

# CEBRA-VISp: kNN on 128D embeddings
pred_cebra, _, acc_cebra = static_image_id_decode(
    emb_visp_128d, visp_train.discrete,
    emb_visp_test_128d, visp_test.discrete,
    bins_per_presentation=bins_per_presentation,
    decoder='knn'
)

print(f'  kNN baseline   : {acc_knn:.1f}%')
print(f'  Naive Bayes    : {acc_bayes:.1f}%')
print(f'  PCA + kNN      : {acc_pca:.1f}%')
print(f'  CEBRA-VISp 128D: {acc_cebra:.1f}%')
print('\nNote: unlike the movie notebook, these values are not meant to match paper percentages because the task is static image identity rather than frame decoding.')

In [ ]:
# Bar chart (Adapted Fig. 4)
# Gray = non-CEBRA baselines, blue = CEBRA

labels = ['kNN\n(raw VISp)', 'Naive Bayes\n(raw VISp)', 'PCA + kNN', 'CEBRA-VISp\n(128D)']
values = [acc_knn, acc_bayes, acc_pca, acc_cebra]
colors = ['#AAAAAA', '#AAAAAA', '#AAAAAA', '#2E75B6']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)

for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f'{val:.1f}%',
        ha='center',
        va='bottom',
        fontweight='bold',
        fontsize=11
    )

ax.set_ylabel('Image decoding accuracy (% correct class)', fontsize=11)
ax.set_title(
    f'Fig. 4 — Static Image Decoding from {cortex}\n'
    f'{num_neurons} neurons, seed={seed}',
    fontsize=12
)
ax.set_ylim(0, 105)
ax.axhline(100 / len(image_names), color='red', linestyle='--', alpha=0.5,
           label=f'Chance ({100 / len(image_names):.1f}%)')
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig4_decoding_bar_chart_static_visp.png', dpi=150, bbox_inches='tight')
plt.show()

**Figure 4 - Static Image Decoding Bar Chart Comparison**  
The bar chart above shows how accurately four different decoding approaches can identify which static image was presented, based on VISp neural activity alone. As in the original replication notebook, the bars progress from raw-activity baselines to a learned CEBRA embedding. The central adapted question is whether the CEBRA-VISp embedding yields more accurate held-out image-identity decoding than the simpler baselines. Because this is a different task from the paper's frame-decoding analysis, the absolute percentages are not expected to match the original Fig. 4 values, but the same general interpretation still applies: stronger decoders indicate that the representation preserves more stimulus information.


### Part 1f — Label-Shuffle Control  [Adapted Fig. 5-style control]

The original replication notebook follows the decoding analysis with an intra- versus inter-area consistency analysis. Because the present notebook uses a single VISp session and one recording modality, the closest control is a **label-shuffle experiment**. We retrain the 128D CEBRA model after shuffling the template feature labels relative to the neural activity. If the original template features carry meaningful stimulus structure, decoding should drop after shuffling.


In [ ]:
# Label-shuffle control (adapted paper-style control)
# Same architecture and hyperparameters as the main 128D CEBRA model, but the
# continuous visual labels are randomly permuted across time.

shuffled_index = visp_train.index[torch.randperm(len(visp_train.index))]

visp_train_shuffle_ds = cebra.data.datasets.TensorDataset(
    visp_train.neural,
    continuous=shuffled_index,
    discrete=visp_train.discrete
)

shuffle_loader = make_loader(visp_train_shuffle_ds)

m_visp_128d_shuffle = single_session_solver(
    shuffle_loader,
    model_architecture='offset1-model',
    distance='cosine',
    num_hidden_units=128,
    output_dimension=128,
    verbose=True,
    device=DEVICE,
    temperature=1,
    learning_rate=3e-4
)
m_visp_128d_shuffle.fit(shuffle_loader)

emb_visp_shuffle_128d = compute_emissions_single(m_visp_128d_shuffle, visp_train_shuffle_ds)
emb_visp_test_shuffle_128d = compute_emissions_single(m_visp_128d_shuffle, visp_test_ds)

pred_shuffle, _, acc_shuffle = static_image_id_decode(
    emb_visp_shuffle_128d, visp_train.discrete,
    emb_visp_test_shuffle_128d, visp_test.discrete,
    bins_per_presentation=bins_per_presentation,
    decoder='knn'
)

print(f'\n── Label-shuffle control ─────────────────────────────────')
print(f'  Real template labels     : {acc_cebra:.1f}%')
print(f'  Shuffled template labels : {acc_shuffle:.1f}%')
print(f'  Difference               : {acc_cebra - acc_shuffle:.1f}%')

In [ ]:
# ── Label-shuffle plot — Adapted Fig. 5 ─────────────────────────────────────────

fig, ax = plt.subplots(figsize=(5, 5))
bars = ax.bar(
    ['Real\nlabels', 'Shuffled\nlabels'],
    [acc_cebra, acc_shuffle],
    color=['steelblue', 'gray'],
    edgecolor='white',
    width=0.6
)

for bar, val in zip(bars, [acc_cebra, acc_shuffle]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f'{val:.1f}%',
        ha='center',
        va='bottom',
        fontweight='bold',
        fontsize=11
    )

ax.set_ylabel('Image decoding accuracy (%)', fontsize=12)
ax.set_ylim(0, 105)
ax.set_title('Fig. 5 — Label-Shuffle Control\n128D CEBRA-Behavior model', fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig5_label_shuffle.png', dpi=150, bbox_inches='tight')
plt.show()

**Figure 5 - Label-Shuffle Control**  
The two bars above compare held-out image decoding from the real 128D CEBRA-VISp model and a control model trained after shuffling the template feature labels. A large drop after shuffling indicates that the continuous template-feature labels are carrying meaningful visual-content information, rather than acting as arbitrary supervision. This is the single-session VISp analogue of the paper-style follow-up analysis: instead of testing cross-area geometric consistency, we test whether the behavior-guided advantage of CEBRA depends on a real relationship between the neural data and the visual labels.


---
## Results Summary

| Figure | Metric | This run |
|--------|--------|----------|
| Fig. 4 | kNN baseline | — |
| Fig. 4 | Naive Bayes | — |
| Fig. 4 | PCA + kNN | — |
| Fig. 4 | CEBRA-VISp (128D) | — |
| Fig. 5 | Real labels | — |
| Fig. 5 | Shuffled labels | — |

Fill in the dashes after running all steps.

**Saved figures:** `p1_01_raw_activity_static_visp.png`, `p1_02_template_feature_tsne.png`, `fig4_embeddings_8D_static_visp.png`, `fig4_decoding_bar_chart_static_visp.png`, `fig5_label_shuffle.png`


In [ ]:
# Final Comparison Reporting
print('=' * 60)
print('RESULTS — STATIC IMAGE VISp ADAPTATION')
print('=' * 60)
print(f'Settings: {cortex}, {num_neurons} neurons, seed={seed}')
print()
print('Fig. 4 — Static Image Decoding (% correct class):')
print(f'  kNN baseline      : {acc_knn:.1f}%')
print(f'  Naive Bayes       : {acc_bayes:.1f}%')
print(f'  PCA + kNN         : {acc_pca:.1f}%')
print(f'  CEBRA-VISp (128D) : {acc_cebra:.1f}%')
print()
print('Fig. 5 — Label-Shuffle Control:')
print(f'  Real labels       : {acc_cebra:.1f}%')
print(f'  Shuffled labels   : {acc_shuffle:.1f}%')

Success updated